## Quantiization: Asymmetric Quantization & Symmetric Quantization 
Lets convert
1) Quantize FP32 to INT16 . Dequantize INT16 back to FP32. Check the error
2) Repeat FP32 to INT8
3) Repeat FP32 to INT4
4) Repeat FP16 to INT16
5) Repeat FP16 to INT8
6) Repeat FP16 to INT4


- For lecture on Asymmetric and Symmetric Quantization: Refer to this video
https://www.youtube.com/watch?v=0VdNflU08yA&t=830s
- This is my own code from scratch.
- But if you'd like to take a look at the corresponding youtube code, its here . See quantization_from_scratch.ipynb https://github.com/hkproj/quantization-notes/tree/main

## Code Notes
i) Code is scratch pad only, hence doesnt cover all edge cases
ii) Assumes that the data that needs to be quantized spans both negative and positive values. Example [-3, 24] etc. Code might not work if lower number is 0 or positive. 


In [1]:
import torch
import math

In [2]:

# Create an tensor of 10 values between -10 and 20
distribution_min = -10
distribution_max =  20
BETA = -10 # quantization lower limit
ALPHA = 20 # quantization upper limit

torch.manual_seed(42) 
x = torch.rand(10)*(distribution_max - distribution_min) + distribution_min
# Convert x to float32
x_fp32 = x.to(torch.float32)
# Convert x to float16
x_fp16 = x.to(torch.float16)
min_value = min(x)
max_value = max(x)

print('x     :', x)
print('x_fp32:', x_fp32)
print('x_fp16:', x_fp16)
print('min = ', min_value, ', max =', max_value)
print('BETA = ',  BETA, ', ALPHA =',ALPHA)



x     : tensor([16.4681, 17.4501,  1.4859, 18.7792,  1.7134,  8.0269, -2.3028, 13.8092,
        18.2231, -6.0044])
x_fp32: tensor([16.4681, 17.4501,  1.4859, 18.7792,  1.7134,  8.0269, -2.3028, 13.8092,
        18.2231, -6.0044])
x_fp16: tensor([16.4688, 17.4531,  1.4863, 18.7812,  1.7139,  8.0234, -2.3027, 13.8125,
        18.2188, -6.0039], dtype=torch.float16)
min =  tensor(-6.0044) , max = tensor(18.7792)
BETA =  -10 , ALPHA = 20


In [3]:
def asymmetric_quantization(xt: torch.Tensor, source_dtype: str, n: int):
    # Upcast float16 to float32 for safe computation
    xt_float = xt.to(torch.float32)

    # Scale
    s = (ALPHA - BETA) / (2**n - 1)

    # Zero point
    z = round(-BETA / s)

    # Quantize
    q = torch.round(xt_float / s) + z
    q = torch.clamp(q, 0, 2**n - 1)

    # Convert to integer type
    if n == 8:
        q = q.to(torch.uint8)
    else:
        q = q.to(torch.uint16)

    # Dequantize
    dq = (q.to(torch.float32) - z) * s
    # Cast dq back to original dtype
    dq = dq.to(xt.dtype)

    print("\n\n")
    print("-----------------------------------------------------------")
    print("Asymmetric Quantization:", source_dtype, "to INT", n)
    print("-----------------------------------------------------------")
    print("Original Tensor:", xt)
    print("Quantized Tensor:", q)
    print("Dequantized Tensor:", dq)

    return q, dq


def symmetric_quantization(xt: torch.Tensor, source_dtype: str, n: int):
    # Upcast float16 to float32 for safe computation
    xt_float = xt.to(torch.float32)

    # Scale
    s = max(abs(ALPHA) ,abs(BETA)) / (2**(n-1) - 1)


    # Quantize
    q = torch.round(xt_float / s) 
    q = torch.clamp(q,-(2**(n-1) - 1), (2**(n-1) - 1))

    # Convert to integer type
    if n == 8:
        q = q.to(torch.int8)
    else:
        q = q.to(torch.int16)

    # Dequantize
    dq = (q.to(torch.float32)) * s
    # Cast dq back to original dtype
    dq = dq.to(xt.dtype)

    print("\n\n")
    print("-----------------------------------------------------------")
    print("Symmetric Quantization:", source_dtype, "to INT", n)
    print("-----------------------------------------------------------")
    print("Original Tensor:", xt)
    print("Quantized Tensor:", q)
    print("Dequantized Tensor:", dq)

    return q, dq



In [9]:

# FP32 to INT16
asymmetric_quantization(x_fp32,'fp32',16)

# FP32 to INT8
asymmetric_quantization(x_fp32,'fp32',8)

# FP32 to INT4
asymmetric_quantization(x_fp32,'fp32',4)

# FP16 to INT16
asymmetric_quantization(x_fp16,'fp16',16)

# FP16 to INT8
asymmetric_quantization(x_fp16,'fp16',8)

# FP16 to INT4
_= asymmetric_quantization(x_fp16,'fp16',4)   





-----------------------------------------------------------
Asymmetric Quantization: fp32 to INT 16
-----------------------------------------------------------
Original Tensor: tensor([16.4681, 17.4501,  1.4859, 18.7792,  1.7134,  8.0269, -2.3028, 13.8092,
        18.2231, -6.0044])
Quantized Tensor: tensor([57820, 59965, 25091, 62868, 25588, 39380, 16814, 52011, 61653,  8728],
       dtype=torch.uint16)
Dequantized Tensor: tensor([16.4683, 17.4502,  1.4859, 18.7791,  1.7134,  8.0270, -2.3030, 13.8091,
        18.2229, -6.0046])



-----------------------------------------------------------
Asymmetric Quantization: fp16 to INT 4
-----------------------------------------------------------
Original Tensor: tensor([16.4688, 17.4531,  1.4863, 18.7812,  1.7139,  8.0234, -2.3027, 13.8125,
        18.2188, -6.0039], dtype=torch.float16)
Quantized Tensor: tensor([13, 14,  6, 14,  6,  9,  4, 12, 14,  2], dtype=torch.uint16)
Dequantized Tensor: tensor([16., 18.,  2., 18.,  2.,  8., -2., 14., 

In [7]:

# FP32 to INT16
symmetric_quantization(x_fp32,'fp32',16)

# FP32 to INT8
symmetric_quantization(x_fp32,'fp32',8)

# FP32 to INT4
symmetric_quantization(x_fp32,'fp32',4)

# FP16 to INT16
symmetric_quantization(x_fp16,'fp16',16)

# FP16 to INT8
symmetric_quantization(x_fp16,'fp16',8)

# FP16 to INT4
symmetric_quantization(x_fp16,'fp16',4)   




-----------------------------------------------------------
Symmetric Quantization: fp32 to INT 16
-----------------------------------------------------------
Original Tensor: tensor([16.4681, 17.4501,  1.4859, 18.7792,  1.7134,  8.0269, -2.3028, 13.8092,
        18.2231, -6.0044])
Quantized Tensor: tensor([26980, 28589,  2434, 30767,  2807, 13151, -3773, 22624, 29856, -9837],
       dtype=torch.int16)
Dequantized Tensor: tensor([16.4678, 17.4499,  1.4856, 18.7793,  1.7133,  8.0270, -2.3029, 13.8090,
        18.2232, -6.0042])



-----------------------------------------------------------
Symmetric Quantization: fp16 to INT 4
-----------------------------------------------------------
Original Tensor: tensor([16.4688, 17.4531,  1.4863, 18.7812,  1.7139,  8.0234, -2.3027, 13.8125,
        18.2188, -6.0039], dtype=torch.float16)
Quantized Tensor: tensor([ 6,  6,  1,  7,  1,  3, -1,  5,  6, -2], dtype=torch.int16)
Dequantized Tensor: tensor([17.1406, 17.1406,  2.8574, 20.0000,  2.8574,

(tensor([ 6,  6,  1,  7,  1,  3, -1,  5,  6, -2], dtype=torch.int16),
 tensor([17.1406, 17.1406,  2.8574, 20.0000,  2.8574,  8.5703, -2.8574, 14.2891,
         17.1406, -5.7148], dtype=torch.float16))

In [6]:
# vectorized naive version
'''
xt is float16 (half precision)
2**16 - 1 = 65535 is way larger than the maximum representable FP16 value, which is around 65504
PyTorch cannot safely store 65535 in float16, so torch.clamp fails with overflow
⚠️ In other words:
float16 cannot hold numbers > 65504
Your INT16 asymmetric quantization wants 0–65535 → overflow error
'''
def asymmetric_quantization_naive(xt: torch.Tensor,source_dtype:str, n:int):
    # Scale should not be a integer
    s = (ALPHA - BETA) / (2**n - 1)
    # Zero point z should be an integer
    z = round(-BETA / s) 

    # Quantize
    q = torch.round(xt / s) + z
    q = torch.clamp(q, 0, 2**n - 1)
    if n == 8:
        q = q.to(torch.uint8)
    else:
        q = q.to(torch.uint16)
        

    # Dequantize
    dq = (q - z) * s

    print("\n\n")
    print("-----------------------------------------------------------")
    print("Asymmetric Quantization:", source_dtype, ' to INT', n)
    print("-----------------------------------------------------------")
    print("Original Tensor:", xt)
    print("Quantized Tensor:", q)
    print("Dequantized Tensor:", dq)

    return q, dq

# FP16 to INT16
asymmetric_quantization_naive(x_fp16,'fp16',16)

RuntimeError: value cannot be converted to type at::Half without overflow